In [1]:
# ── Cell 1: Imports ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from pathlib import Path
from jiwer import wer, cer, transforms as tr
import warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path("/home/ridwan/projects/afrispeech-project")
PREDICTIONS_DIR = PROJECT_ROOT / "results" / "predictions"
METRICS_DIR = PROJECT_ROOT / "results" / "metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# Imports your existing text_normalization.py from src/
import sys
sys.path.append(str(PROJECT_ROOT / "src"))

from text_normalization import normalize

In [10]:
clinical_df = pd.read_csv(PREDICTIONS_DIR / "whisper_small_clinical_predictions.csv")
general_df  = pd.read_csv(PREDICTIONS_DIR / "whisper_small_general_predictions.csv")

In [16]:
general_df.describe()

,row_index,duration,error
count,2712.000000,2712.000000,0.0
mean,1355.500000,10.797843,NaN
std,783.031289,12.541619,NaN
min,0.000000,1.003991,NaN
25%,677.750000,5.650493,NaN
50%,1355.500000,8.996984,NaN
75%,2033.250000,13.502993,NaN
max,2711.000000,301.867004,NaN


In [17]:
clinical_df.describe()

,row_index,duration,error
count,3606.000000,3606.000000,0.0
mean,1802.500000,10.530936,NaN
std,1041.106863,12.176854,NaN
min,0.000000,1.000000,NaN
25%,901.250000,5.000000,NaN
50%,1802.500000,8.000000,NaN
75%,2703.750000,12.482494,NaN
max,3605.000000,271.181000,NaN


In [21]:
# Filter both sets to the correct duration range
# Paper: 2–17 seconds at collection time
# Your data has outliers up to 301s — filter them out

print("=== Before filtering ===")
print(f"General: {len(general_df)} samples")
print(f"Clinical: {len(clinical_df)} samples")

general_df_clean = general_df[
    (general_df["duration"] >= 2.0) & 
    (general_df["duration"] <= 30.0)
].reset_index(drop=True)

clinical_df_clean = clinical_df[
    (clinical_df["duration"] >= 2.0) & 
    (clinical_df["duration"] <= 30.0)
].reset_index(drop=True)

print("\n=== After filtering ===")
print(f"General: {len(general_df_clean)} samples (removed {len(general_df) - len(general_df_clean)})")
print(f"Clinical: {len(clinical_df_clean)} samples (removed {len(clinical_df) - len(clinical_df_clean)})")

# Check max duration after filtering
print(f"\nGeneral max duration: {general_df_clean['duration'].max():.1f}s")
print(f"Clinical max duration: {clinical_df_clean['duration'].max():.1f}s")

=== Before filtering ===
General: 2712 samples
Clinical: 3606 samples

=== After filtering ===
General: 2619 samples (removed 93)
Clinical: 3446 samples (removed 160)

General max duration: 29.7s
Clinical max duration: 30.0s


In [22]:
# Recompute corpus-level WER on clean data
refs_gen  = [normalize(r) for r in general_df_clean["reference"].tolist()]
hyps_gen  = [normalize(p) for p in general_df_clean["prediction"].tolist()]

refs_clin = [normalize(r) for r in clinical_df_clean["reference"].tolist()]
hyps_clin = [normalize(p) for p in clinical_df_clean["prediction"].tolist()]

print("=== Whisper-Small — Filtered Results ===")
print(f"General  WER: {wer(refs_gen,  hyps_gen):.3f}")
print(f"General  CER: {cer(refs_gen,  hyps_gen):.3f}")
print()
print(f"Clinical WER: {wer(refs_clin, hyps_clin):.3f}")
print(f"Clinical CER: {cer(refs_clin, hyps_clin):.3f}")

=== Whisper-Small — Filtered Results ===
General  WER: 0.377
General  CER: 0.225

Clinical WER: 0.468
Clinical CER: 0.252


In [23]:
# Test 1: WER with NO normalisation at all
raw_wer_gen = wer(
    general_df_clean["reference"].tolist(),
    general_df_clean["prediction"].tolist()
)
print(f"General WER (no normalisation): {raw_wer_gen:.3f}")

# Test 2: WER with lowercase only
refs_lower = [r.lower() for r in general_df_clean["reference"].tolist()]
hyps_lower = [p.lower() for p in general_df_clean["prediction"].tolist()]
print(f"General WER (lowercase only):   {wer(refs_lower, hyps_lower):.3f}")

# Test 3: WER with BasicTextNormalizer only (no spoken punct removal)
from transformers.models.whisper.english_normalizer import BasicTextNormalizer
basic_norm = BasicTextNormalizer()

refs_basic = [basic_norm(r) for r in general_df_clean["reference"].tolist()]
hyps_basic = [basic_norm(p) for p in general_df_clean["prediction"].tolist()]
print(f"General WER (BasicNormalizer):  {wer(refs_basic, hyps_basic):.3f}")

# Test 4: Your full normaliser (current)
refs_full = [normalize(r) for r in general_df_clean["reference"].tolist()]
hyps_full = [normalize(p) for p in general_df_clean["prediction"].tolist()]
print(f"General WER (full normaliser):  {wer(refs_full, hyps_full):.3f}")

General WER (no normalisation): 0.508
General WER (lowercase only):   0.482
General WER (BasicNormalizer):  0.442
General WER (full normaliser):  0.377


In [24]:
print("=== Duration distribution ===")
for label, df in [("General", general_df), ("Clinical", clinical_df)]:
    total = len(df)
    under_2   = (df["duration"] < 2).sum()
    btw_2_17  = ((df["duration"] >= 2) & (df["duration"] <= 17)).sum()
    btw_17_30 = ((df["duration"] > 17) & (df["duration"] <= 30)).sum()
    over_30   = (df["duration"] > 30).sum()
    
    print(f"\n{label} (total: {total})")
    print(f"  < 2s:      {under_2}")
    print(f"  2–17s:     {btw_2_17}")
    print(f"  17–30s:    {btw_17_30}")
    print(f"  > 30s:     {over_30}")
    print(f"  Paper test: {'2,723' if label == 'General' else '3,623'}")

=== Duration distribution ===

General (total: 2712)
  < 2s:      53
  2–17s:     2314
  17–30s:    305
  > 30s:     40
  Paper test: 2,723

Clinical (total: 3606)
  < 2s:      59
  2–17s:     3055
  17–30s:    391
  > 30s:     101
  Paper test: 3,623


In [25]:
general_df_clean = general_df[
    (general_df["duration"] >= 2.0) &
    (general_df["duration"] <= 30.0)
].reset_index(drop=True)

clinical_df_clean = clinical_df[
    (clinical_df["duration"] >= 2.0) &
    (clinical_df["duration"] <= 30.0)
].reset_index(drop=True)

print(f"General:  {len(general_df_clean)} (paper: 2,723 | gap: {2723 - len(general_df_clean)})")
print(f"Clinical: {len(clinical_df_clean)} (paper: 3,623 | gap: {3623 - len(clinical_df_clean)})")

General:  2619 (paper: 2,723 | gap: 104)
Clinical: 3446 (paper: 3,623 | gap: 177)


In [26]:
# ── Load small.en predictions ─────────────────────────────────────────────────
general_en_df = pd.read_csv(
    PREDICTIONS_DIR / "whisper_small_en_general_predictions.csv"
)
clinical_en_df = pd.read_csv(
    PREDICTIONS_DIR / "whisper_small_en_clinical_predictions.csv"
)

# ── Apply final duration filter ───────────────────────────────────────────────
general_en_clean = general_en_df[
    (general_en_df["duration"] >= 2.0) &
    (general_en_df["duration"] <= 30.0)
].reset_index(drop=True)

clinical_en_clean = clinical_en_df[
    (clinical_en_df["duration"] >= 2.0) &
    (clinical_en_df["duration"] <= 30.0)
].reset_index(drop=True)

print(f"General clean:  {len(general_en_clean)} samples")
print(f"Clinical clean: {len(clinical_en_clean)} samples")

FileNotFoundError: [Errno 2] No such file or directory: '/home/ridwan/projects/afrispeech-project/results/predictions/whisper_small_en_general_predictions.csv'

In [27]:
# ── Load predictions ──────────────────────────────────────────────────────────
clinical_en_df = pd.read_csv(
    PREDICTIONS_DIR / "whisper_small_en_clinical_predictions.csv"
)
general_en_df = pd.read_csv(
    PREDICTIONS_DIR / "whisper_small_en_general_predictions.csv"
)

# ── Apply final duration filter (2–30s) ───────────────────────────────────────
clinical_en_clean = clinical_en_df[
    (clinical_en_df["duration"] >= 2.0) &
    (clinical_en_df["duration"] <= 30.0)
].reset_index(drop=True)

general_en_clean = general_en_df[
    (general_en_df["duration"] >= 2.0) &
    (general_en_df["duration"] <= 30.0)
].reset_index(drop=True)

print(f"Clinical clean: {len(clinical_en_clean)} samples")
print(f"General clean:  {len(general_en_clean)} samples")

Clinical clean: 3446 samples
General clean:  2619 samples


In [28]:
# ── Normalise ─────────────────────────────────────────────────────────────────
refs_clin_en = [normalize(r) for r in clinical_en_clean["reference"].tolist()]
hyps_clin_en = [normalize(p) for p in clinical_en_clean["prediction"].tolist()]

refs_gen_en  = [normalize(r) for r in general_en_clean["reference"].tolist()]
hyps_gen_en  = [normalize(p) for p in general_en_clean["prediction"].tolist()]

In [29]:
# ── Corpus-level metrics ──────────────────────────────────────────────────────
clin_en_wer = wer(refs_clin_en, hyps_clin_en)
clin_en_cer = cer(refs_clin_en, hyps_clin_en)
clin_en_wrr = 1 - clin_en_wer

gen_en_wer  = wer(refs_gen_en,  hyps_gen_en)
gen_en_cer  = cer(refs_gen_en,  hyps_gen_en)
gen_en_wrr  = 1 - gen_en_wer

print("=== Whisper-Small-EN — WER/CER/WRR (2–30s) ===")
print(f"{'Domain':<12} {'N':>6} {'WER':>8} {'CER':>8} {'WRR':>8} {'Paper WER':>10}")
print("-" * 56)
print(f"{'Clinical':<12} {len(clinical_en_clean):>6} {clin_en_wer:>8.3f} {clin_en_cer:>8.3f} {clin_en_wrr:>8.3f} {'0.482':>10}")
print(f"{'General':<12} {len(general_en_clean):>6} {gen_en_wer:>8.3f} {gen_en_cer:>8.3f} {gen_en_wrr:>8.3f} {'0.350':>10}")

=== Whisper-Small-EN — WER/CER/WRR (2–30s) ===
Domain            N      WER      CER      WRR  Paper WER
--------------------------------------------------------
Clinical       3446    0.484    0.245    0.516      0.482
General        2619    0.377    0.223    0.623      0.350


In [32]:
baseline_small = pd.DataFrame([
    {
        "model": "openai/whisper-small",
        "model_short": "whisper-small",
        "domain": "clinical",
        "n_samples": len(clinical_df_clean),
        "WER": round(clin_en_wer, 3),
        "CER": round(clin_en_cer, 3),
        "WRR": round(1 - clin_en_wer, 3),
        "paper_WER": 0.455,
        "duration_filter": "2-30s"
    },
    {
        "model": "openai/whisper-small",
        "model_short": "whisper-small",
        "domain": "general",
        "n_samples": len(general_df_clean),
        "WER": round(gen_en_wer, 3),
        "CER": round(gen_en_cer, 3),
        "WRR": round(1 - gen_en_wer, 3),
        "paper_WER": 0.330,
        "duration_filter": "2-30s"
    }
])

METRICS_DIR.mkdir(parents=True, exist_ok=True)
baseline_small.to_csv(
    METRICS_DIR / "whisper_small_baseline_final.csv", index=False
)

display(baseline_small)
print("Saved.")

,model,model_short,domain,n_samples,WER,CER,WRR,paper_WER,duration_filter
0,openai/whisper-small,whisper-small,clinical,3446,0.484,0.245,0.516,0.455,2-30s
1,openai/whisper-small,whisper-small,general,2619,0.377,0.223,0.623,0.330,2-30s


Saved.
